<a href="https://colab.research.google.com/github/k2herat/Empathetic_AI/blob/base_models_5_classes/res_mob_dense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
path ="/kaggle/input/human-face-emotions/Data"

In [ ]:
import os, random, json
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tqdm import tqdm

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
def get_dataloaders(data_dir, input_size=224, batch_size=32, val_split=0.15, test_split=0.15):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(input_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    eval_tf = transforms.Compose([
        transforms.Resize(int(input_size*1.14)),
        transforms.CenterCrop(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    full = datasets.ImageFolder(data_dir, transform=train_tf)
    classes = full.classes
    n = len(full)


    val_n = int(n * val_split)
    test_n = int(n * test_split)
    train_n = n - val_n - test_n


    train_set, val_set, test_set = random_split(full, [train_n, val_n, test_n])


    val_set.dataset.transform = eval_tf
    test_set.dataset.transform = eval_tf


    loaders = {
    'train': DataLoader(train_set, batch_size=batch_size, shuffle=True),
    'val': DataLoader(val_set, batch_size=batch_size, shuffle=False),
    'test': DataLoader(test_set, batch_size=batch_size, shuffle=False)
    }
    return loaders, classes

In [ ]:
def build_model(name, num_classes):
    name = name.lower()
    if name == 'resnet18':
        m = models.resnet18(pretrained=True)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == 'mobilenet_v2':
        m = models.mobilenet_v2(pretrained=True)
        m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, num_classes)
    elif name == 'densenet121':
        m = models.densenet121(pretrained=True)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    else:
        raise ValueError("Unknown model")
    return m.to(device)

In [ ]:
def train_one_epoch(model, loader, criterion, opt):
    model.train()
    total_loss = 0
    correct = 0
    n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        opt.step()
        total_loss += loss.item()*x.size(0)
        correct += (out.argmax(1)==y).sum().item()
        n += x.size(0)
    return total_loss/n, correct/n

In [ ]:
def eval_model(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    n = 0
    preds = []
    labels = []
    probs_list = []
    softmax = nn.Softmax(dim=1)
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item()*x.size(0)
            correct += (out.argmax(1)==y).sum().item()
            n += x.size(0)
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.cpu().numpy())
            probs_list.append(softmax(out).cpu().numpy())
    return total_loss/n, correct/n, np.array(preds), np.array(labels), np.vstack(probs_list)

In [ ]:
## Train Models

loaders, classes = get_dataloaders(path, batch_size=32)
num_classes = len(classes)
models_to_train = ["resnet18", "mobilenet_v2", "densenet121"]
trained_paths = {}


for name in models_to_train:
    model = build_model(name, num_classes)
    opt = optim.Adam(model.parameters(), lr=1e-4)
    crit = nn.CrossEntropyLoss()
    best_acc = 0
    for epoch in range(5):
        tl, ta = train_one_epoch(model, loaders['train'], crit, opt)
        vl, va, _, _, _ = eval_model(model, loaders['val'], crit)
        print(name, epoch, tl, ta, vl, va)
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), f"{name}_best.pth")

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 160MB/s]


resnet18 0 0.791099807634672 0.6907495588697397 0.6361567040212748 0.7629738267148014
resnet18 1 0.44214877226500454 0.8402020739165116 0.4485307096466691 0.8431859205776173
resnet18 2 0.21792840209612027 0.9256967440961059 0.40157579377105307 0.8793998194945848
resnet18 3 0.125100658456472 0.9593435014865486 0.4183789648465301 0.8866200361010831
resnet18 4 0.0974344219792264 0.9675859901863624 0.4247011319609756 0.8956453068592057


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 120MB/s]


mobilenet_v2 0 0.8688470326425812 0.6547098208890285 0.6862767322614305 0.7378158844765343
mobilenet_v2 1 0.602390683053734 0.7706847791931546 0.5873527030329412 0.7822653429602888
mobilenet_v2 2 0.41185527028644386 0.8500398830098378 0.48361810888516776 0.8326940433212996
mobilenet_v2 3 0.266152888073257 0.9066254139372991 0.4530253497217967 0.8604467509025271
mobilenet_v2 4 0.1771121224072554 0.9384834787653187 0.44161553635171175 0.8757897111913358


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 155MB/s]


densenet121 0 0.8157417080731496 0.6815643808464866 0.6370092491595754 0.7518050541516246
densenet121 1 0.5193408445607605 0.8079814362717845 0.5056256243037833 0.8163357400722022
densenet121 2 0.3270316078355938 0.8849193879770854 0.415862208642469 0.8623646209386282
densenet121 3 0.1988475907175522 0.9325372845713181 0.4145447807890844 0.8694720216606499
densenet121 4 0.1460310427067644 0.9502066665055232 0.40300645351087144 0.8872969314079422


In [ ]:
def load_for_inference(name, num_classes):
    model = build_model(name, num_classes)
    model.load_state_dict(torch.load(f"{name}_best.pth", map_location=device))
    model.eval()
    return model

In [ ]:

models = [load_for_inference(n, num_classes) for n in models_to_train]
softmax = nn.Softmax(dim=1)
all_probs = []
for m in models:
    _, _, _, labels, pb = eval_model(m, loaders['test'], nn.CrossEntropyLoss())
    all_probs.append(pb)
    avg_probs = np.mean(np.stack(all_probs), axis=0)
    preds = np.argmax(avg_probs, axis=1)


print("Accuracy:", accuracy_score(labels, preds))
print(classification_report(labels, preds, target_names=classes))
print(confusion_matrix(labels, preds))



Accuracy: 0.9200135379061372
              precision    recall  f1-score   support

       Angry       0.90      0.90      0.90      1580
        Fear       0.90      0.83      0.86      1471
       Happy       0.97      0.97      0.97      2745
         Sad       0.87      0.91      0.89      1854
     Suprise       0.94      0.95      0.95      1214

    accuracy                           0.92      8864
   macro avg       0.92      0.91      0.91      8864
weighted avg       0.92      0.92      0.92      8864

[[1417   42   16   96    9]
 [  63 1226   21  119   42]
 [  22   14 2665   35    9]
 [  56   69   24 1696    9]
 [   9   17   24   13 1151]]
